# World of Shadow Work — full offline factory (Colab Pro+, A100)

Run on a **GPU A100** runtime (Runtime → Change runtime type → A100), then **Runtime → Run all**.

This builds the whole asset bundle from scratch — corpus → CLIP embeddings → UMAP galaxy + k-NN
webs → image-to-3D **vessel** keyframes (LGM) → packed `vessel.wswv` — and pushes the assets back
to GitHub so a `git pull` on your Mac picks them up (relaunch, no code change).

Edit the **CONFIG** cell first. Generating one vessel per image (~700+) takes a couple of hours on
the A100 — set `VESSEL_LIMIT = 24` for a quick smoke-test, then 0 (= all) for the full run.

In [ ]:
# ===== CONFIG =====
GH_TOKEN   = ""        # GitHub token (repo scope) to push assets back. Blank => download manually.
REPO       = "9LiveZZZ-Git/MAT201B_Projects"
BRANCH     = "world-of-shadow-work"

CORPUS_PER_QUERY = 20      # images per search term per source (bigger => richer galaxy)
VESSEL_MODE      = "all"   # "all" = one vessel per image; "reps" = one per cluster (fast)
VESSEL_LIMIT     = 0       # cap vessel inputs (0 = no cap; try 24 first to smoke-test)
VESSEL_G         = 8000    # gaussians per vessel keyframe (stage_d caps the total bank size)

In [ ]:
!nvidia-smi -L

In [ ]:
import os
_auth = (GH_TOKEN + "@") if GH_TOKEN else ""
!git clone -b {BRANCH} https://{_auth}github.com/{REPO}.git
%cd MAT201B_Projects/reagency/factory

In [ ]:
!pip -q install open_clip_torch umap-learn faiss-cpu hdbscan scikit-learn plyfile pillow

In [ ]:
# Build a fresh, larger corpus (images are gitignored, so fetch them here).
!python3 fetch_corpus.py --per-query {CORPUS_PER_QUERY}

In [ ]:
!python3 stage_a_embed.py        # CLIP ViT-L/14 image+text embedding (A100)

In [ ]:
!python3 stage_b_layout.py       # UMAP galaxy + kNN webs + clusters + cluster_reps + atlas -> ../assets

In [ ]:
!python3 prep_vessel_inputs.py --mode {VESSEL_MODE} --limit {VESSEL_LIMIT}

In [ ]:
# LGM image-to-3D (3DTopia/LGM). torch + CUDA are preinstalled on Colab.
!pip -q install -U xformers
![ -d diff-gaussian-rasterization ] || git clone --recursive https://github.com/ashawkey/diff-gaussian-rasterization
!pip -q install ./diff-gaussian-rasterization
!pip -q install git+https://github.com/NVlabs/nvdiffrast
![ -d LGM ] || git clone https://github.com/3DTopia/LGM
!cd LGM && pip -q install -r requirements.txt
!mkdir -p LGM/pretrained
![ -f LGM/pretrained/model_fp16.safetensors ] || wget -q -O LGM/pretrained/model_fp16.safetensors https://huggingface.co/ashawkey/LGM/resolve/main/model_fp16_fixrot.safetensors

In [ ]:
import os
inp = os.path.abspath("work/vessel_inputs")
out = os.path.abspath("work/vessels")
os.makedirs(out, exist_ok=True)
%cd LGM
!python infer.py big --resume pretrained/model_fp16.safetensors --workspace {out} --test_path {inp}
%cd ..
!echo "generated:" && ls work/vessels | wc -l

In [ ]:
!python3 stage_d_vessel.py --plys work/vessels --G {VESSEL_G}
!ls -lh ../assets

In [ ]:
# Push the generated assets so your Mac can `git pull` them. (Needs GH_TOKEN in CONFIG.)
!git -C .. add assets
!git -C .. -c user.email="wosw@colab" -c user.name="wosw-colab" commit -m "assets: galaxy + vessels (Colab)"
!git -C .. push
# No token? download instead:  from google.colab import files; files.download('../assets/vessel.wswv')